In [1]:
import sys, os

import jax
import jax.numpy as jnp
import numpy as np

import wavefunctions, hamiltonian, trajectory, optimization, qc
from pyscf import gto, scf, ao2mo, cc

jax.config.update("jax_enable_x64", False)

import time, importlib
import matplotlib.pyplot as plt

/home/amress/miniforge3/envs/nqs/lib/python3.9/site-packages/flax/struct.py:132: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(data_clz, keypaths)
/home/amress/miniforge3/envs/nqs/lib/python3.9/site-packages/flax/struct.py:132: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(data_clz, keypaths)
No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


In [16]:
print(jnp.linalg.norm(1j))
print(jnp.linalg.norm(1))
print(jnp.linalg.norm(1+1j))

1.0
1.0
1.4142135


In [25]:
print(jnp.eye(5,3))
print(jnp.eye(3,5))

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 0.]
 [0. 0. 0.]]
[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]]


In [24]:
class LogMPSlater(wavefunctions.Wavefunction):
    """
    Creates a log-wavefunction that is a Slater determinant built from a
    multiple planewave basis.
    """
    N: int
    dim: int
    kpoints: jnp.ndarray          # (Nk, dim)
    init_coeffs: jnp.ndarray      # (N, Nk) or (N, Nk) complex

    def setup(self):
        
        if self.dim not in (2, 3):
            raise ValueError("Only dim=2 or dim=3 are supported.")

        Nk = self.kpoints.shape[0]
        if self.init_coeffs.shape != (self.N, Nk):
            raise ValueError(f"Expected init_coeffs shape (N, Nk)={(self.N, Nk)} "
                             f"but got {self.init_coeffs.shape}")

        self.C = self.param("MP_coefficients", lambda rng: self.init_coeffs)

    def __call__(self, rs):
        
        eikrs = jnp.exp(1j * (self.kpoints @ rs.T))   # ( Nk , N )
        slaterMatrix = self.C @ eikrs                 # ( N , N )
        return jnp.linalg.slogdet(slaterMatrix)[1]

r_ws = 20
N = 56
dim = 2
numKpoints = 169

walkers = 1

NUp = N // 2
NDown = N - NUp
spins = ( NUp , NDown )

lattice = wavefunctions.computeLattice(
    N, r_ws, dim, basis_matrix=jnp.array([[7,0],[0,4*jnp.sqrt(3)]])
)
kpoints = wavefunctions.genKpoints(numKpoints, lattice, dim)
init_coeffs = jnp.eye(numKpoints, spins[0]) + 0.1j * np.random.normal(size=(numKpoints, spins[0]))

wavefunction = wavefunctions.LogSlaterCYJastrow(spins, dim, lattice, kpoints)
wavefunction = LogMPSlater(spins[0], dim, kpoints, init_coeffs)

rng = jax.random.PRNGKey(558)

rng, rs_rng, init_rng = jax.random.split(rng, 3)
rs = trajectory.wignerCrystal(
    spins, lattice, r_ws, walkers, rs_rng, dim=dim, gridShape=(7,4)
)[:,:spins[0],:]

print(kpoints.shape)
print(rs.shape)

parameters = wavefunction.initBatch(init_rng, rs)
wavefunction.applyBatch(parameters, rs)

(169, 2)
(1, 28, 2)


Array([47.979324], dtype=float32)